# Stage 1: PneumoniaMNIST calibration and controlled shift

This notebook is the first reproducible experiment in the trustworthy medical imaging pilot.

**Research question:** does a model's confidence remain meaningful when the input images are changed?

The controlled contrast transformation is a stress test, not a real hospital-to-hospital shift.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from medmnist import PneumoniaMNIST

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
transform = transforms.ToTensor()
train_ds = PneumoniaMNIST(split='train', download=True, transform=transform, size=64)
val_ds = PneumoniaMNIST(split='val', download=True, transform=transform, size=64)
test_ds = PneumoniaMNIST(split='test', download=True, transform=transform, size=64)

print('Train:', len(train_ds))
print('Validation:', len(val_ds))
print('Test:', len(test_ds))
print('Task:', train_ds.info['task'])
print('Labels:', train_ds.info['label'])

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1)
        )
        self.classifier = nn.Linear(64, 1)

    def forward(self, x):
        return self.classifier(self.features(x).flatten(1)).squeeze(1)

model = SmallCNN().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=128)
    test_loader = DataLoader(test_ds, batch_size=128)

## Next experiment

Train the baseline model, collect validation/test logits, and then add calibration metrics and the controlled contrast shift.

The test set must not be used to fit the temperature parameter.